Creates a very basic de novo simulated dataset, with a more realistic ratio of true positive to true negative than `simple_spread` provides.

This is similar to `simple_de_novo_simulation.ipynb`

# Imports

In [2]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd
 

%load_ext autoreload
%autoreload 2

2025-09-12 17:05:34.480854: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-12 17:05:34.485273: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

# Create cluster

In [3]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='24GB')
client=Client(cluster)

# Seting experimiental design parameters

In [4]:
#first, we define the new parameters we want to assign to this object.

new_cell_number=pd.Series({"reference":1000,"blood":2000,"neuron":1000})

#we make up 3x replicates
new_zi=pd.Series({"replicate_A":0.02,"replicate_B":0.021})

new_min=1
new_max=200

new_MOI=60

In [5]:
#next, let's create a bounds object from these parameters and the bounds of the shendure data.
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi,
    cells_per_cell_type=new_cell_number)
    
artificial_bounds.set_effective_moi(new_MOI)

# Creating an artificial library

`cat shendure_counts_grouped.txt | cut -f4,11 | grep "^minP" | cut -f2 | awk '{ sum += $1; n++ } END { if (n > 0) print sum / n }'`
Produces `0.0727759` as the average MPRA UMIs per cell for minP.

In [6]:
import math

In [7]:
def alpha_for_expected_groups(n, K_target):
    """
    Helper for sample_crp_groups
    Choose alpha so E[K_n] ~= K_target using H_n ≈ log n + gamma.
    Technically only valid for large n, but good enough for our purposes
    """
    gamma = 0.5772156649015329
    return max(1e-12, K_target / (math.log(n) + gamma))

def sample_crp_groups(n, alpha, rng=None):
    """
    Chinese Restaurant Process partition of n items.
    Returns: np.array of length n with group ids in 0..K-1.
    """
    if rng is None:
        rng = np.random.default_rng()
    groups = np.full(n, -1, dtype=int)
    # Track current group sizes
    sizes = []  # list of counts per existing group
    for i in range(n):
        # Prob of joining existing group k is sizes[k] / (alpha + i)
        # Prob of creating new group is alpha / (alpha + i)
        total = alpha + i
        if len(sizes) == 0 or rng.random() < alpha / total:
            # new group
            sizes.append(1)
            groups[i] = len(sizes) - 1
        else:
            # join existing: pick proportional to sizes
            k = rng.choice(len(sizes), p=np.array(sizes) / (total - alpha))
            sizes[k] += 1
            groups[i] = k
    return groups

In [35]:
from typing import List
import itertools


def activity_spread(cell_types:List[str],
    minimum:float,
    maximum:float,
    minp_value:float,
    total:int,
    frac_active:int,
    ct_specificity:float):
    """
    Creates a ground-truth dataframe of an scMPRA experiment
    with a controllable number of active CREs.
    (see readme for ground truth dataframe specification)
    Assumes experiment is interested in activity vs a known negative control,
    not skew. 
    
    This is useful to create datasets with a balance 
    of active and inactive CREs which roughly
    approx. real libraries. 

    `total` is the total number of CREs to create.
    `frac_active` is the fraction of the library that is active elements
    `ct_specificity` is how much cell-type specificity there is. Its 
    how many different values a given CRE will take across different cell-types. So 
    specificity=2 implies that, on average, there will be two different 
    means for each CRE across cell-types. 
    """
    #first, we create the reference minP control
    reference=pd.DataFrame({"cell_type":cell_types})
    reference["true_mean"]=minp_value
    reference["cre_id"]="reference"

    #second, we create a DF of all the inactive CREs.
    total_inactive =int(total*(1-frac_active))
    inactive_names=[f"inactive_{i}"for i in range(0,total_inactive)]
    inactive_tuples=[i for i in itertools.product(inactive_names,cell_types)]
    inactive=pd.DataFrame(inactive_tuples,columns=["cre_id","cell_type"])
    inactive["true_mean"]=minp_value
    
    #third, we create a df of active elements.
    total_active=int(total*frac_active)
    active_names=[f"active_{i}"for i in range(0,total_active)]
    active=pd.DataFrame(active_names,columns=["cre_id"])
    active["cell_types"]=[cell_types for _ in range(len(active))]
    #we have a list of all cell types in each.
    #now we want to randomly decide which cell_types are the same
    #and which are different. 
    alpha = alpha_for_expected_groups(n=len(cell_types), K_target=ct_specificity)
    def _apply_crp(row):
        groups = sample_crp_groups(n=len(cell_types), alpha=alpha)
        return groups

    active["groups"]=active.apply(_apply_crp,axis=1)

    #calculate and print the percent of active CREs with any cell-type specificity
    def _different_groups(row):
        return len(set(row["groups"]))==1
    is_cell_type_specific=spread_gt.apply(_different_groups,axis=1)
    print(f"{sum(is_cell_type_specific)/len(is_cell_type_specific)*100}% of active elements are not cell-type specific.")
    
    final_gt=pd.concat([reference,inactive])
    return (active, None)

    #hypotheses are "all test CRE vs - ctrl"
    #and "all same-CRE different-cell-type comparisons"

#making up the CREs
spread_gt,spread_hypothesis=activity_spread(
    cell_types=list(new_cell_number.keys()),
    minimum=new_min,
    maximum=new_max,
    minp_value=0.0727759,
    total=1000,
    frac_active=0.5,
    ct_specificity=.2)
spread_gt

82.39999999999999% of active elements are not cell-type specific.


,cre_id,cell_types,groups
0,active_0,"[reference, blood, neuron]","[0, 0, 0]"
1,active_1,"[reference, blood, neuron]","[0, 0, 0]"
2,active_2,"[reference, blood, neuron]","[0, 0, 0]"
3,active_3,"[reference, blood, neuron]","[0, 0, 0]"
4,active_4,"[reference, blood, neuron]","[0, 0, 0]"
...,...,...,...
495,active_495,"[reference, blood, neuron]","[0, 0, 0]"
496,active_496,"[reference, blood, neuron]","[0, 0, 0]"
497,active_497,"[reference, blood, neuron]","[0, 0, 0]"
498,active_498,"[reference, blood, neuron]","[0, 0, 0]"


53.0% of active elements are cell-type specific.


In [26]:
n_cell_types=3
ct_specificity=0.1


In [31]:
minimum=new_min
maximum=new_max
def _generate_means(row):
    uniq_groups=list(set(row["groups"]))
    means=np.random.uniform(low=minimum,
        high=maximum,
        size=len(uniq_groups)
    )
    means=means.tolist()
    means_dict=dict(zip(uniq_groups,means))
    means_rep=[means_dict[i] for i in row["groups"]]
    return means_rep

spread_gt["means"]=spread_gt.apply(_generate_means,axis=1)
spread_gt

,cre_id,cell_types,groups,means
0,active_0,"[reference, blood, neuron]","[0, 0, 0]","[44.11326871602343, 44.11326871602343, 44.1132..."
1,active_1,"[reference, blood, neuron]","[0, 0, 0]","[162.20516287905278, 162.20516287905278, 162.2..."
2,active_2,"[reference, blood, neuron]","[0, 0, 0]","[19.300872811156918, 19.300872811156918, 19.30..."
3,active_3,"[reference, blood, neuron]","[0, 0, 0]","[58.104529791979594, 58.104529791979594, 58.10..."
4,active_4,"[reference, blood, neuron]","[0, 0, 0]","[18.289964324521513, 18.289964324521513, 18.28..."
...,...,...,...,...
495,active_495,"[reference, blood, neuron]","[0, 0, 0]","[168.20605043534215, 168.20605043534215, 168.2..."
496,active_496,"[reference, blood, neuron]","[0, 1, 0]","[83.62492490815208, 3.503066927996076, 83.6249..."
497,active_497,"[reference, blood, neuron]","[0, 0, 0]","[180.79558432963134, 180.79558432963134, 180.7..."
498,active_498,"[reference, blood, neuron]","[0, 0, 0]","[88.9347027985064, 88.9347027985064, 88.934702..."


In [34]:
spread_gt=spread_gt.drop(columns="groups")

In [35]:
spread_gt.explode(["cell_types","means"])

,cre_id,cell_types,means
0,active_0,reference,44.113269
0,active_0,blood,44.113269
0,active_0,neuron,44.113269
1,active_1,reference,162.205163
1,active_1,blood,162.205163
...,...,...,...
498,active_498,blood,88.934703
498,active_498,neuron,88.934703
499,active_499,reference,141.346881
499,active_499,blood,141.346881


In [5]:
new_cell_number.keys()

Index(['reference', 'blood', 'neuron'], dtype='object')